# RAG 接続の実機確認（Issue #7・Colab 用）

v2 の同期パイプラインから v1 の検索（実 Qdrant）と生成（実 vLLM）を呼び、次を確かめる。データは架空の社内規程（evaluation/corpus）だけ。トークン不要（使うモデルは全て公開）。ランタイムは **T4 GPU**。

1. 2 問続けて回答できる（共有ループの使い回し）
2. 権限外の文書が出ない（v1 の生の検索結果を記録して照合）／2b 上位ユーザーには上位文書が出る（対照）
3. 監査に本文が残らない（監査 DB の行とファイルの生バイトの両方を見る）
4. vLLM・Qdrant を固める／落とすと保留で止まり、戻せば回答に戻る
5. 遅延（参考値。T4 は本番の 6GB 機より速いので合否には使わない）

- 「Run all」で全セルを流す。最後のセルに `総合 PASS/FAIL` と結果 JSON（本文なし）が出る
- 埋め込み・リランカーは本番と同じく CPU で動く（GPU は vLLM だけ）
- vLLM が起動しないときは 3 番目のセルの `MODEL` を予備のモデルに替えて 4 番目のセルからやり直す

In [ ]:
# 依存の衝突を避けるため、vLLM と RAG 側を別の仮想環境に入れる
!pip -q install uv
!git clone -q --depth 1 -b redesign/v2 https://github.com/Maaa2005/local-rag-agent.git repo
!uv venv -q /content/rag-env && uv pip install -q --python /content/rag-env/bin/python \
    --extra-index-url https://download.pytorch.org/whl/cpu torch "transformers<5" sentence-transformers==3.3.1 \
    qdrant-client==1.12.1 aiosqlite==0.20.0 pydantic-settings==2.6.1 openai==1.57.0 httpx==0.28.1 pyyaml
!uv venv -q /content/vllm-env && uv pip install -q --python /content/vllm-env/bin/python vllm

In [ ]:
MODEL = "cyankiwi/gemma-4-E2B-it-AWQ-INT4"   # 本番の既定
EXTRA_ARGS = ["--quantization", "awq"]
# 予備（AWQ / Gemma 4 が T4 で動かない場合）:
# MODEL = "sbintuitions/sarashina2.2-3b-instruct-v0.1"; EXTRA_ARGS = []
GPU_UTIL = 0.55          # 埋め込み等は CPU なので GPU は vLLM だけが使う
FAULT_TIMEOUT = 15       # 止める試験で待つ秒数
QDRANT_VER = "v1.12.6"   # qdrant-client 1.12.1 に合わせる

In [ ]:
import subprocess, time, urllib.request, pathlib, os

def launch(argv, log, env=None):
    # 新しいプロセスグループで起動する（止める試験で killpg を使うため。pid = pgid）
    f = open(log, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)

def wait(url, proc, log, limit):
    t = time.time()
    while time.time() - t < limit:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(url, timeout=3)
            print("ready:", url, f"{time.time() - t:.0f} 秒")
            return
        except Exception:
            time.sleep(3)
    print(pathlib.Path(log).read_text()[-4000:])
    raise RuntimeError(f"起動しない: {url}（上はログの末尾）")

qd = pathlib.Path("/content/qdrant"); (qd / "bin").mkdir(parents=True, exist_ok=True)
!curl -sSL https://github.com/qdrant/qdrant/releases/download/{QDRANT_VER}/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C /content/qdrant/bin
q_env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH=str(qd / "storage"), QDRANT__SERVICE__HTTP_PORT="6333")
qdrant = launch([str(qd / "bin" / "qdrant")], "/content/qdrant.log", q_env)
wait("http://127.0.0.1:6333/readyz", qdrant, "/content/qdrant.log", 60)

In [ ]:
vllm = launch(["/content/vllm-env/bin/vllm", "serve", MODEL, "--served-model-name", "local-llm", "--port", "8001",
               "--dtype", "half", "--max-model-len", "4096", "--gpu-memory-utilization", str(GPU_UTIL), *EXTRA_ARGS],
              "/content/vllm.log")
wait("http://127.0.0.1:8001/v1/models", vllm, "/content/vllm.log", 1200)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# 止める試験で両サーバーを止めて戻す。最後に落とす試験をするので、この後サーバーは止まっている
!/content/rag-env/bin/python repo/evaluation/colab/rag_connection_check.py --repo repo --work /content/rag_work \
    --qdrant-pgid {qdrant.pid} --vllm-pgid {vllm.pid} --fault-timeout {FAULT_TIMEOUT} --rerank on --label colab-T4

In [ ]:
import json
r = json.loads(pathlib.Path("/content/rag_work/rag_check_result.json").read_text())
r["model"] = MODEL
print(json.dumps(r, ensure_ascii=False, indent=1))